## EDA & Dataförståelse


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)

### Läser in datan och visar de 10 översta raderna

In [ ]:
df = pd.read_csv("historical_data.csv")
df.head(10)

### Inläsning av datastorleken, datatyper och target-fördelning

In [ ]:
print("Datasetstorlek (rader, kolumner):", df.shape)
print("\nDatatyper:")
print(df.dtypes)
print("\nSammanfattning:")
print(df.info())

Datasetet innehåller 12.000st rader och 18st kolumner. Target-variabeln är is_suspicious som indikerar om en händelse i datasetet verkar misstänkt eller inte. Variabeln är av typen int64, som gör att värdet 0 innebär att händelsen inte är misstänkt, medans värdet 1 innebär att den är misstänkt.

### Fördelning av misstänkta värden i antal och procent

In [ ]:
target_col = "is_suspicious"

print("Misstänkta värden (antal):")
print(df[target_col].value_counts())

print("\nMisstänkta värden (%):")
print(df[target_col].value_counts(normalize=True)* 100)

sns.set_style("whitegrid")

plt.figure(figsize=(8,5))

ax = sns.countplot(
    x=df[target_col],
    hue=df[target_col],
    palette="mako",
    legend=False
    )

plt.title("Fördelning av misstänkta värden")
plt.xlabel("Misstänkta värden")
plt.ylabel("Antal")
plt.show()

#### Slutsats:
Datasetet är obalanserat eftersom att majoriteten av observationerna, 89%, inte är misstänkta. Det är en mindre del, 10.2%, som kan klassificeras som misstänkta.

### Saknade värden

In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0]

sns.set_style("whitegrid")

plt.figure(figsize=(8,5))

ax = sns.barplot(
    x=missing.index,
    y=missing.values,
    hue=missing.index,
    palette="mako",
    legend=False
)

plt.title("Saknade värden per variabel")
plt.xlabel("Variabel")
plt.ylabel("Antal")
plt.show()

print("Saknade värden per kolumn:")
print(missing)

#### Slutsats: 
Analysen visar att datasetet innehåller en mindre andel saknade värden i variablerna region, price och time_to_first_response_min. Största andelen finns i price. 
Eftersom att andelen saknade värden är relativt liten i förhållande till storleken på hela datasetet, är det inte nödvändigt att ta bort dem.
Dessa värden hanteras istället i preprocessing-steget genom att ersätta med medianvärdet för varje variabel, eftersom medianen påverkas mindre av extremvärden.

## Train/test + preprocessing


### Uppdelning av features och target

Separerar features (X) och target-variabeln (y)  
Target är *is_suspicious* som anger om en händelse är misstänkt (1) eller inte (0)  
Kolumnen *id* tas bort från feature-setet eftersom den endast fungerar som en unik identifierare för varje rad och inte innehåller någon meningsfull information för prediktion. Att inkludera den skulle kunna leda till att modellen lär sig slumpmässiga mönster istället för faktiska samband i datan.

In [ ]:
# Separerar features (X) och target (y)
X = df.drop(columns=[target_col, 'id'])
y = df[target_col]

### Train/Test-split

Delar upp datan i tränings- och testmängd (80/20) för att kunna utvärdera modellen på osedd data.

Använder stratifierad split (stratify=y) för att bevara klassfördelningen mellan misstänkta och icke-misstänkta händelser i båda mängderna.

Kontrollerar datamängdernas storlek samt klassfördelning för att verifiera att splitten genomförts korrekt.

In [ ]:
# Delar upp i 80% träning och 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Kontroll av dimensioner och klassfördelning
print(f'Train: {X_train.shape, y_train.shape}')
print(f'Test:  {X_test.shape, y_test.shape}')

print(f'\nKlassfördelning Train \n{y_train.value_counts(normalize=True)}')
print(f'\nKlassfördelning Test \n{y_test.value_counts(normalize=True)}')

### Identifiering av variabeltyper

För att kunna tillämpa rätt preprocessing delar vi upp våra features i två grupper: numeriska och kategoriska.

(Inkluderar både object och string för att säkerställa att alla textbaserade variabler fångas upp oavsett Pandas-version.)

**Verifiering**: Enligt vår EDA förväntar vi oss totalt 16 features (18 kolumner minus 'target' - 'id'). Vi kontrollerar att summan stämmer för att säkerställa att ingen data har fallit bort.

In [ ]:
# Identifierar kolumner baserat på datatyp
numeric_features = X.select_dtypes(include=['number']).columns
categorical_features = X.select_dtypes(include=['object', 'string']).columns

# Kontroll av antal för att matcha EDA
print(f'Antal numeriska: {len(numeric_features)}')
print(f'Antal kategoriska: {len(categorical_features)}')
print(f'Total feautures: {len(numeric_features) + len(categorical_features)}')

print(f'\nNumeriska variabler: \n{numeric_features}')
print(f'\nKategoriska variabler: \n{categorical_features}')

### Preprocessing Pipeline

**Numerisk hantering**: Vi använder median för imputation för att minimera påverkan från outliers, följt av StandardScaler för att normalisera skalorna.

**Kategorisk hantering**: Vi fyller saknade värden med det mest förekommande värdet (most_frequent) och omvandlar text till binära kolumner via OneHotEncoder.

**Driftsäkerhet**: Parametern handle_unknown='ignore' inkluderas för att hantera nya kategorier i framtida data utan att pipelinen avbryts.

**Exkludering av övrig data**: Med remainder='drop' säkerställer vi att endast de features vi valt ut (de i numeric_features och categorical_features) skickas vidare till modellen

In [ ]:
# Pipeline för siffror
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Pipeline för text/kategorier
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Samlar alla transformationer i ett objekt
preprocess = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder= 'drop'
)

## Modellering och jämförelse

DEN HÄR TEXTEN SKA TAS BORT ALT. SKRIVA EN SAMMANFATTNING?

Skapa en baseline.

Träna minst två ytterligare modeller (minst 3 totalt inkl baseline).

Utvärdera med rimlig metod (t.ex. cross-validation på train eller tydligt valideringsupplägg)

Välj metric och motivera valet utifrån ert kravkort.

(Exempel på modeller: LogisticRegression, DecisionTree, RandomForest, GradientBoosting…)

DEN HÄR TEXTEN SKA TAS BORT ALT. SKRIVA EN SAMMANFATTNING?

Länka till dokumentationen?

### Motivering till val av metric

Vi har två kostnader att jobba med: granskningstid och missade bedrägerier. 

* Hög Precision sparar granskningstid, men innebär fler missade bedrägerier
* Hög Recall minskar missade bedrägerier, men innebär högre granskningstid

F1-score är det harmoniska medelvärdet mellan Precision och Recall och ger oss möjligheten att ta hänsyn till båda dessa kostnader då F1-score blir låg om antingen Precision eller Recall är låg. 

### Skapa baseline och modeller

Vi väljer DummyClassifier med strategy = "stratified" som baseline. Vi väljer "stratified" istället för "most_frequent" för att "stratified" blir mer ärlig när vi använder F1-score för att jämföra våra modeller. 

DummyClassifier med "most_frequent" kommer alltid gissa is_not_suspicious vilket ger ett F1-score = 0 vilket inte blir så intressant att jämföra med. 

"stratified" gissar slumpmässigt, men vet hur stor andel av träningsdatan som är is_suspicious och ser till att gissa utifrån den fördelningen.

DummyClassifier med strategy="stratified" blir därför den mest realistiska "dumma" modellen att jämföra med. 

<br>

https://scikit-learn.org/stable/modules/generated/sklearn.dummy.DummyClassifier.html

In [ ]:
# Skapa baseline

baseline = DummyClassifier(strategy="stratified", random_state=42)

### Modeller att utvärdera

* LogisticRegression: snabb, linjär 
* DecisionTree: icke-linjär, bygger på flödesschema 
* RandomForest: ensemble-modell dvs en hel skog med DecisionTrees 

Vi valde dessa tre modeller för att få en bredd i vårt test. Med dessa modeller kan vi testa både linjärt vs icke-linjärt och enskild modell vs ensemble. 

#### Val av hyperparametrar inför test

Vi vill inte laborera med en massa olika hyperparametrar innan vi testar olika modeller för första gången. 

Att lägga för mycket tid och energi på hyperparametrarna i det här läget kan medföra att en modell får "mer kärlek" än en annan och att testresultatet därmed inte blir lika tillförlitigt.

Dock kan vi inte helt och hållet gå på default, utan måste anpassa modellen lite efter den träningsdata vi faktiskt har: 

* LogisticRegression
    * max_iter=1000, default är 100. =1000 ger modellen tillräckligt tid att räkna
    * class_weight="balanced", default är att klasserna är lika vanliga, men vi har obalanserad data. Med class_weight kan man styra straffvikten utifrån klassfördelningen i datan 
* DecisionTree
    * DecisionTree overfit:ar gärna. För att resultatet på ett rimligt sätt ska kunna jämföras mot andra modeller behövs hyperparametrar för att "stoppa upp" modellen i dess plugghästiga eftersträvan att svara rätt på varenda studiefråga
    * Dessa hyperparametrar stoppar upp modellen:
        * max_depth=5, default är None. 
        * min_samples_leaf=5, default är 2.
* RandomForest
    * Precis som med de två andra modellerna behöver vi anpassa modellen efter den obalanserade datan. Detta görs med class_weight:
        * class_weight="balanced_subsample", default är None. Med class_weight="balanced_subsample" beräknas straffvikten utifrån klassfördelningen i varje träd 

<br>


https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html

https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html

https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html

In [ ]:
# Skapar de tre modellerna som ska testas

logreg = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)

tree = DecisionTreeClassifier(max_depth=5, min_samples_leaf=5, class_weight="balanced", random_state=42)

rf = RandomForestClassifier(random_state=42, class_weight="balanced_subsample")

### Skapa pipelines

In [ ]:
# Skapar pipelines för att koppla ihop vår preprocessing med varje modell 
# Detta gör vi för att förhindra dataläckage

pipe_baseline = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", baseline)
])

pipe_logreg = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", logreg)
])

pipe_tree = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", tree)
])

pipe_rf = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", rf)
])

### Utvärdera modellerna

#### Metod för utvärdering

Vi väljer att utvärdera de olika modellerna med Cross-Validation. 

I Cross-Validation delas träningsdatan upp i olika "tårtbitar" (vanligtvis 5). Sedan tränas och testas datan i olika rundor (antal rundor = antal tårtbitar). 

Detta gör så att all data blir testdata någon gång och resultatet därmed blir mer tillförlitligt. 

Med StratifiedKFold bestämmer vi hur datan ska delas upp. Vi väljer 5 "tårtbitar" och sätter shuffle till True (för en mer rättvis fördelning). Eftersom vi vet att datan är obalaserad är det _Stratified_ KFold vi använder oss av. 

Själva "magin" sker med funktionen cross_val_score. 

cross_val_score: 

- delar upp datan utifrån vår StratifiedKFold och kopierar modellen
- tränar kopian och gör sedan en predict
- gissningen jämförs mot target och F1-score beräknas 
- sedan gör samma sak på nästa tårtbit 

Vi lägger in cross_val_score i en for-loop för att kunna göra det på alla 4 modeller. 

Resultatet lägger vi i en lista som vi sedan gör om till en DataFrame för att ge en tydligare tabell.

<br>

https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedKFold.html

https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html 

https://scikit-learn.org/stable/modules/cross_validation.html#cross-validation 

In [ ]:
models = {
    "Baseline": pipe_baseline, 
    "LogisticRegression": pipe_logreg,
    "DecisionTree": pipe_tree,
    "RandomForest": pipe_rf
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
SCORING = "f1"

results = []

for name, model in models.items():
    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=SCORING
    )
    results.append({
        "model": name,
        "mean_f1": scores.mean(),
        "std": scores.std()
    })

results_df = pd.DataFrame(results).sort_values("mean_f1", ascending=False)
results_df.reset_index(drop=True, inplace=True)
results_df.index += 1

print("Resultat av Cross-Validation:\n")
print(results_df)

best_model_name = results_df.iloc[0]["model"]
best_model_score = results_df.iloc[0]["mean_f1"]

print(f"\nDen bästa modellen är: {best_model_name} med F1-score {best_model_score:.3f}")

## Välj och optimera EN modell

DEN HÄR TEXTEN SKA TAS BORT ALT. SKRIVA EN SAMMANFATTNING? Länka till dokumentation?

Välj en “final” modell baserat på jämförelsen.

Gör tuning på den valda modellen (litet grid, minst 1–2 parametrar).

Förklara kort vad ni optimerade och varför (koppla till kravkortet).

DEN HÄR TEXTEN SKA TAS BORT ALT. SKRIVA EN SAMMANFATTNING? Länka till dokumentation?

### Hyperparameter-tuning 


Efter modelljämförelsen visade **Logistic Regression** högst F1-score och valdes därför som slutmodell.

Eftersom vårt kravkort fokuserar på **kostnader och tradeoffs mellan granskningstid och missade bedrägerier**, är det viktigt att modellen både kan identifiera misstänkta fall och samtidigt undvika för många felaktiga flaggningar.

För att förbättra modellens prestanda genomförde vi därför en **hyperparameter-optimering med GridSearchCV**.

GridSearchCV testar olika kombinationer av parametrar och använder cross-validation för att hitta den kombination som ger bäst resultat enligt vald metric (F1-score).

Vi testade följande parametrar:

**C**  
Regulariseringsstyrkan i Logistic Regression.  
Ett lägre C-värde innebär starkare regularisering (en enklare modell), medan ett högre värde gör modellen mer flexibel.

**solver**  
Algoritmen som används för att optimera modellen under träning.

**class_weight**  
Används för att hantera obalanserad data genom att ge högre vikt till den mindre klassen, vilket är viktigt eftersom endast cirka 10 % av händelserna är misstänkta.

**imputer strategy**  
Vi testade både *median* och *mean* för att se vilken metod som fungerar bäst för att hantera saknade numeriska värden.

Totalt testades flera kombinationer av dessa parametrar med **Stratified 5-fold cross-validation**.

Den kombination som gav högst F1-score valdes som slutmodell och används i de efterföljande analyserna, där vi analyserar olika threshold-värden för att hitta en rimlig balans mellan **granskningskostnad och risken att missa bedrägerier**.

In [ ]:
# Modell med högst f1-score från utvärderingen

model = logreg

# Pipeline

final_pipeline = pipe_logreg

# Hyperparametrar att testa

param_grid = {
    "model__C": [0.0008, 0.001, 0.01, 0.1, 1, 10],
    "model__solver": ["lbfgs", "liblinear"],
    "model__class_weight": [None, "balanced"],
    "preprocess__num__imputer__strategy": ["median", "mean"]
}

# GridSearch för att hitta bästa parameter

grid_search = GridSearchCV(
    estimator=final_pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring=SCORING,
    n_jobs=-1,
    refit=True,
    return_train_score=True
)

# Träna modellerna

grid_search.fit(X_train, y_train)

# Bästa modellen

best_model = grid_search.best_estimator_

print("Bästa parametrar från GridSearch:\n")
for param_name, param_value in grid_search.best_params_.items():
    print(f"{param_name}: {param_value}")

print(f"\nBäst F1-score: {grid_search.best_score_:.3f}")

result = pd.DataFrame(grid_search.cv_results_)
print("\nSammanställning av GridSeach och korsvalidering (topp 5 kombinationer):")
result.sort_values("rank_test_score").head(5)

### Resultat av hyperparameter-optimering

GridSearchCV testade flera kombinationer av hyperparametrar för Logistic Regression.

Den bästa kombinationen valdes baserat på **mean_test_score (F1-score)** från cross-validation.

Vi jämförde även **mean_train_score** och **mean_test_score** för att kontrollera att modellen inte överanpassar träningsdatan.  
Resultaten visar att skillnaden mellan tränings- och testscore är relativt liten, vilket tyder på att modellen generaliserar bra till ny data.

Den optimerade modellen används sedan i threshold-analysen där vi analyserar hur olika tröskelvärden påverkar:

- antal ärenden som behöver granskas
- antal misstänkta fall som missas

Detta gör det möjligt att tydligt visa **tradeoffen mellan kostnad och risk**, vilket är centralt för den stakeholder vi pitchar till.

## Threshold / prioritering (kopplat till kravkortet)


### Val av prioriteringsstrategi

Vi använder **Alternativ A)** Threshold-beslut, där en händelse flaggas som misstänkt om modellens sannolikhet överstiger ett valt tröskelvärde. Genom att justera threshold kan vi styra hur många händelser som flaggas:

**Lägre threshold:** fler flaggade händelser → färre misstänkta missas, men mer granskning krävs.

**Högre threshold:** färre flaggade händelser → mindre granskning, men fler misstänkta missas.


### Modellens standardprediktion (utgångspunkt)  
Som utgångspunkt använder vi modellens default threshold 0.5, vilket innebär att en händelse klassificeras som misstänkt om sannolikheten ≥ 0.5. Confusion matrix nedan visar modellens prestation på testdatan med detta threshold:

- **True Positives (TP)** – misstänkta händelser som korrekt flaggas
- **False Positives (FP)** – oskyldiga händelser som flaggas i onödan
- **False Negatives (FN)** – misstänkta händelser som modellen missar
- **True Negatives (TN)** – korrekta icke-misstänkta prediktioner

Denna matris fungerar som en **utgångspunkt** för vår vidare analys. Därefter undersöker vi hur resultaten förändras när vi justerar threshold-värdet och därmed prioriteringsstrategin.

In [ ]:
y_pred = best_model.predict(X_test)

cm = confusion_matrix(y_test, y_pred)

target_names = ["ej misstänkt", "misstänkt"]

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)

disp.plot()
plt.title("Confusion matrix")
plt.show()

### Modellens prestanda med standard-threshold (0.5)

Av de **2155 ej misstänkta** händelserna klassificeras **1608 korrekt**, medan **547 felaktigt** flaggas som misstänkta.  
Av de **245 misstänkta** händelserna identifieras **143 korrekt**, medan **102 missas** av modellen.

Detta innebär att modellen har hög accuracy, men låg recall för misstänkta händelser, vilket betyder att många misstänkta fall inte fångas vid detta threshold.

Nedan beräknar vi några vanliga prestationsmått för modellen:  
- Accuracy – andel korrekta prediktioner totalt  
- Precision – hur stor andel av de flaggade fallen som faktiskt är misstänkta  
- Recall – hur stor andel av de verkligt misstänkta fallen som modellen hittar  
- F1-score – ett mått som väger ihop precision och recall

Vi skriver även ut ett classification report för att få en mer detaljerad bild av modellens resultat.

In [ ]:
test_results = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score"],
    "Score": [
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred, zero_division=0),
        recall_score(y_test, y_pred, zero_division=0),
        f1_score(y_test, y_pred)
    ]
})

print(test_results)
print()

print("Classification report: \n", classification_report(y_test, y_pred))

### Analys av olika threshold-värden

Istället för att enbart använda modellens standardthreshold (0.5) analyserar vi hur modellens beteende förändras vid olika thresholds.

Vi använder modellens predikterade sannolikheter (predict_proba) och testar tre threshold-värden: 0.4, 0.6 och 0.8.

För varje threshold beräknar vi:  
- flagged_cases – antal händelser som flaggas för granskning  
- false_positives – oskyldiga händelser som flaggas i onödan  
- false_negatives – misstänkta händelser som modellen missar  
- precision – andel av de flaggade fallen som faktiskt är misstänkta  
- recall – andel av de verkligt misstänkta fallen som modellen identifierar

Syftet är att analysera hur valet av threshold påverkar balansen mellan granskningstid och risken att missa misstänkta händelser

In [ ]:
# Prediktera sannolikheter

y_proba = best_model.predict_proba(X_test)[:,1]

thresholds = [0.4, 0.6, 0.8]

results = []

for t in thresholds:

    y_pred = (y_proba >= t).astype(int)

    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

    results.append({
        "threshold": t,
        "flagged_cases": y_pred.sum(),
        "false_positives": fp,
        "false_negatives": fn,
        "precision": precision,
        "recall": recall
    })

results_df = pd.DataFrame(results)

results_df

### Visualisering av tradeoff
För att se den fullständiga bilden av hur kostnaderna rör sig mellan dessa punkter har vi också tagit fram en tradeoff-graf. 
Den hjälper oss att identifiera den punkt där vi får bäst balans mellan operativt arbete och risk.

In [ ]:
# Skapa intervall av thresholds för en jämnare graf
threshold_range = np.linspace(0.1, 0.9, 50)
fps = []
fns = []

for t in threshold_range:
    y_pred_t = (y_proba >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_t).ravel()
    fps.append(fp)
    fns.append(fn)

plt.figure(figsize=(8, 5))
plt.plot(threshold_range, fps, label='Onödig granskning (False Positives)', color='orange', lw=2)
plt.plot(threshold_range, fns, label='Missade bedrägerier (False Negatives)', color='red', lw=2)

# Marker vald threshold (0.6)
plt.axvline(x=0.6, color='black', linestyle='--', alpha=0.5)
plt.text(0.61, 1000, 'Rekommenderat: 0.6', fontweight='bold')

plt.title('Tradeoff: Granskningskostnad vs. Risk', fontsize=14)
plt.xlabel('Threshold', fontsize=12)
plt.ylabel('Antal fall i testdatan', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### Tolkning

Resultatet visar tydligt hur valet av threshold påverkar modellens beteende.

En **lägre threshold** (0.4) gör att många **fler händelser flaggas** (1569). Detta gör att modellen fångar de flesta misstänkta fall (hög recall), men leder också till många false positives, vilket innebär **mer arbete för granskningsteamet.**

En **högre threshold** (0.8) gör tvärtom att väldigt **få händelser flaggas** (12). Precisionen blir högre, men modellen **missar nästan alla misstänkta fall** (mycket låg recall).

Threshold 0.6 hamnar mellan dessa två ytterligheter. Den flaggar 320 händelser, vilket minskar granskningstiden jämfört med 0.4, samtidigt som modellen fortfarande hittar en del misstänkta fall.

Grafen indikerar att balansen mellan granskningskostnad och risk ligger runt threshold 0.63.  
**Vi rekommenderar dock att använda 0.60**. Genom att välja 0.60 istället för 0.63 fångar vi fler bedrägerier utan att det innebär någon större ökning av arbetsbördan för teamet.


## Deploy-test: ny data (tisdag kursvecka 6)

När ni får new_data.csv ska ni:
använda er låsta pipeline
skapa prediktioner och en prioriteringslista